# Week 5 Exercise: Personal Knowledge Base RAG

## Approach: "Digital twin" over your own knowledge

This notebook implements a **minimal RAG system** you can chat with:
- Uses the course **Insurellm knowledge base** (company, employees, products, contracts) so it runs without extra setup.
- You can later point it at your own folder of markdown/docs for a personal knowledge assistant.
- Idea inspiration from: https://medium.com/@yashwanths_29644/rag-series-02-demo-to-build-rag-using-open-source-frameworks-4e381e84b992

**What we do:**
1. Load documents from a knowledge base (by folder → doc type).
2. Chunk text and store embeddings in Chroma.
3. Run a conversational RAG chain (retrieve → generate) with memory.
4. Expose a Gradio chat UI.

**Tech:** LangChain • OpenAI Embeddings • Chroma • ConversationalRetrievalChain • Gradio  
**Requires:** `.env` with `OPENAI_API_KEY`. Run from repo root or set `KNOWLEDGE_BASE_PATH` to your docs folder.

## 1. Imports and configuration

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
import gradio as gr
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from sklearn.manifold import TSNE

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate, ChatPromptTemplate

MODEL = "gpt-4o-mini"
DB_NAME = "sammyloto_rag_db"
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150

load_dotenv(override=True)
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")

In [ ]:
# Resolve knowledge base path (works from repo root, week5/, or sammyloto/)
def get_knowledge_base_path():
    base = Path.cwd()
    candidates = [
        base / "knowledge-base",        # run from week5/
        base / "week5" / "knowledge-base",  # run from repo root
        base / ".." / ".." / "knowledge-base",  # run from sammyloto/
    ]
    for p in candidates:
        try:
            resolved = p.resolve()
            if resolved.exists() and resolved.is_dir():
                return resolved
        except Exception:
            continue
    return base / ".." / ".." / "knowledge-base"

KNOWLEDGE_BASE_PATH = get_knowledge_base_path()
print(f"Knowledge base path: {KNOWLEDGE_BASE_PATH}")
print(f"Exists: {KNOWLEDGE_BASE_PATH.exists()}")

## 2. Load and chunk documents

In [ ]:
def add_metadata(doc, doc_type):
    doc.metadata["doc_type"] = doc_type
    return doc

documents = []
text_loader_kwargs = {"encoding": "utf-8"}

if KNOWLEDGE_BASE_PATH.exists():
    for folder in sorted(KNOWLEDGE_BASE_PATH.iterdir()):
        if folder.is_dir():
            doc_type = folder.name
            loader = DirectoryLoader(
                str(folder), glob="**/*.md", loader_cls=TextLoader, loader_kwargs=text_loader_kwargs
            )
            folder_docs = loader.load()
            documents.extend([add_metadata(doc, doc_type) for doc in folder_docs])
else:
    print("Knowledge base not found. Create a folder with subfolders (e.g. company/, employees/) and add .md files.")

text_splitter = CharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = text_splitter.split_documents(documents) if documents else []

print(f"Loaded {len(documents)} documents → {len(chunks)} chunks")
if documents:
    print(f"Doc types: {sorted(set(d.metadata['doc_type'] for d in documents))}")

## 3. Vector store (Chroma)

In [ ]:
if not chunks:
    raise SystemExit("No chunks to embed. Check KNOWLEDGE_BASE_PATH.")

embeddings = OpenAIEmbeddings()

if os.path.exists(DB_NAME):
    Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(
    documents=chunks, embedding=embeddings, persist_directory=DB_NAME
)
print(f"Vectorstore created: {vectorstore._collection.count()} vectors")

## 3b. Vector visualization (t-SNE, inspo from Hope Ogbons)

Reduce high-dimensional embeddings to 2D with t-SNE so we can see how chunks cluster by document type (company, employees, products, contracts). Points that are close in the plot are semantically similar.

In [ ]:
collection = vectorstore._collection
result = collection.get(include=["embeddings", "documents", "metadatas"])
vectors = np.array(result["embeddings"])
doc_texts = result["documents"]
doc_types = [m["doc_type"] for m in result["metadatas"]]

doc_list = sorted(set(doc_types))
color_palette = px.colors.qualitative.Plotly
color_map = [color_palette[i % len(color_palette)] for i in range(len(doc_list))]
colors = [color_map[doc_list.index(t)] for t in doc_types]

tsne = TSNE(n_components=2, random_state=42)
reduced = tsne.fit_transform(vectors)

fig = go.Figure(data=[go.Scatter(
    x=reduced[:, 0],
    y=reduced[:, 1],
    mode="markers",
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>{d[:80]}..." for t, d in zip(doc_types, doc_texts)],
    hoverinfo="text",
)])
fig.update_layout(
    title="Knowledge base chunks in 2D (t-SNE)",
    xaxis_title="t-SNE 1",
    yaxis_title="t-SNE 2",
    width=800,
    height=600,
)
fig.show()

## 4. Conversational RAG chain

In [ ]:
system_template = """You are a knowledgeable, friendly expert on Insurellm. You answer only from the provided context (company docs, employees, products, contracts).

HOW YOU COMMUNICATE:
- Be concise and accurate; sound human, not robotic.
- If the answer isn't in the context, say so gracefully (e.g. "From what I have in the knowledge base, that isn't covered—but here's what I do know...").
- Never make up facts; only use the context you're given.

Context:
{context}

Chat history:
{chat_history}"""

llm = ChatOpenAI(temperature=0.3, model_name=MODEL)
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True, output_key="answer")
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
    combine_docs_chain_kwargs={
        "prompt": ChatPromptTemplate.from_messages([
            SystemMessagePromptTemplate.from_template(system_template),
            HumanMessagePromptTemplate.from_template("{question}"),
        ])
    },
)

## 5. Chat UI (Gradio)

In [ ]:
def chat(message, history):
    if not message or not message.strip():
        return ""
    result = qa_chain.invoke({"question": message})
    return result.get("answer", "No response.")

demo = gr.ChatInterface(
    fn=chat,
    title="Week 5 RAG: Insurellm knowledge base",
    description="Ask about company, employees, products, or contracts.",
)
demo.launch(inbrowser=True)